#Camada Gold

In [0]:
%sql
CREATE OR REPLACE TABLE spotify.gold.artist_discography AS

SELECT
  ar.artist_id,
  ar.name AS artist_name,
  ar.image_url,
  COUNT(DISTINCT al.album_id) AS total_releases,
  COUNT(t.track_id) AS total_tracks,
  MIN(CAST(LEFT(al.release_date, 4) AS INT)) AS first_release_year,
  MAX(CAST(LEFT(al.release_date, 4) AS INT)) AS last_release_year,
  MAX(CAST(LEFT(al.release_date, 4) AS INT))
    - MIN(CAST(LEFT(al.release_date, 4) AS INT)) AS career_years,
  ROUND(AVG(t.music_duration_ms) / 60000, 2) AS avg_track_minutes,
  current_timestamp() AS _processed_at
FROM spotify.silver.artists ar
LEFT JOIN spotify.silver.albums al ON al.artist_id = ar.artist_id
LEFT JOIN spotify.silver.album_tracks t ON t.album_id = al.album_id
GROUP BY ar.artist_id, ar.name, ar.image_url;

SELECT * FROM spotify.gold.artist_discography

In [0]:
%sql
CREATE OR REPLACE TABLE spotify.gold.duration_by_year AS

SELECT
  ar.artist_id,
  ar.name AS artist_name,
  COUNT(t.track_id) AS total_tracks,
  EXTRACT(YEAR FROM al.release_date) AS year_release,
  ROUND(AVG(t.music_duration_ms) / 60000, 2) AS avg_track_minutes,
  current_timestamp() AS _processed_at
FROM spotify.silver.artists ar
LEFT JOIN spotify.silver.albums al ON al.artist_id = ar.artist_id
LEFT JOIN spotify.silver.album_tracks t ON t.album_id = al.album_id
GROUP BY ar.artist_id, ar.name, year_release
ORDER BY ARTIST_ID, year_release ASC; 

SELECT * FROM spotify.gold.duration_by_year

In [0]:
%sql
CREATE OR REPLACE TABLE spotify.gold.albums_summary AS

WITH BASE AS (
    SELECT
    ar.artist_id,
    ar.name AS artist_name,
    al.album_id,
    al.album_name,
    al.image_url,
    EXTRACT(YEAR FROM al.release_date) AS year_release,
    ROUND(SUM(t.music_duration_ms) / 60000, 2) AS total_album_minutes,
    current_timestamp() AS _processed_at
    FROM spotify.silver.artists ar
    LEFT JOIN spotify.silver.albums al ON al.artist_id = ar.artist_id
    LEFT JOIN spotify.silver.album_tracks t ON t.album_id = al.album_id
    GROUP BY ar.artist_id, ar.name, al.album_id, al.album_name, al.image_url, year_release
    ORDER BY ARTIST_ID, year_release ASC 
)
SELECT
    artist_id,
    artist_name,
    album_id,
    album_name,
    image_url,
    year_release,
    total_album_minutes,
    _processed_at
FROM BASE;

SELECT * FROM spotify.gold.albums_summary